# PGD Attack & Adversarial Training on MNIST

This notebook demonstrates the **Projected Gradient Descent (PGD)** attack and
**adversarial training** defense using the `sleight` package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sleight.data import get_dataset
from sleight.models import get_mnist_cnn_model
from sleight.attacks import pgd_attack
from sleight.defenses import adversarial_train
from sleight.evaluation import evaluate_robustness

In [ ]:
# Load data
(x_train, y_train), (x_test, y_test) = get_dataset("mnist", one_hot=True)
y_train_int = np.argmax(y_train, axis=1)
y_test_int = np.argmax(y_test, axis=1)

In [ ]:
# Train a standard model
model = get_mnist_cnn_model()
model.fit(x_train, y_train, epochs=3, batch_size=64, verbose=1)

In [ ]:
# Evaluate standard model under PGD attack
epsilon = 0.15
x_sub = x_test[:200]
y_sub_int = y_test_int[:200]

results_std = evaluate_robustness(model, x_sub, y_sub_int, pgd_attack, epsilon, alpha=0.01, num_iter=10)
print("Standard model:")
print(f"  Clean accuracy:       {results_std['clean_accuracy']:.4f}")
print(f"  Adversarial accuracy: {results_std['adversarial_accuracy']:.4f}")

In [ ]:
# Adversarial training with PGD
robust_model = get_mnist_cnn_model()
robust_model = adversarial_train(
    robust_model,
    x_train[:5000],
    y_train_int[:5000],
    pgd_attack,
    epsilon=0.15,
    epochs=3,
    batch_size=64,
)

In [ ]:
# Evaluate adversarially trained model
results_robust = evaluate_robustness(robust_model, x_sub, y_sub_int, pgd_attack, epsilon, alpha=0.01, num_iter=10)
print("Adversarially trained model:")
print(f"  Clean accuracy:       {results_robust['clean_accuracy']:.4f}")
print(f"  Adversarial accuracy: {results_robust['adversarial_accuracy']:.4f}")

In [ ]:
# Compare side by side
labels = ['Clean Acc', 'Adversarial Acc']
std_vals = [results_std['clean_accuracy'], results_std['adversarial_accuracy']]
robust_vals = [results_robust['clean_accuracy'], results_robust['adversarial_accuracy']]

x_pos = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_pos - width/2, std_vals, width, label='Standard')
ax.bar(x_pos + width/2, robust_vals, width, label='Adversarially Trained')
ax.set_ylabel('Accuracy')
ax.set_title('Standard vs Adversarially Trained Model')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()